In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import mlflow
import mlflow.pyfunc
import joblib
import os


# Load champion model + features
champion = joblib.load("/content/drive/MyDrive/olist_project/model/champion.pkl")
FEATURES = joblib.load("/content/drive/MyDrive/olist_project/model/features.pkl")

# Point MLflow to Drive
mlflow_path = "/content/drive/MyDrive/olist_project/mlflow"
mlflow.set_tracking_uri(f"file://{mlflow_path}")
mlflow.set_experiment("olist-late-delivery")

print(" Champion model loaded!")
print(f"   Features: {len(FEATURES)}")
print(f"   MLflow URI: {mlflow.get_tracking_uri()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


 Champion model loaded!
   Features: 30
   MLflow URI: file:///content/drive/MyDrive/olist_project/mlflow


In [5]:
class LateDeliveryPipeline(mlflow.pyfunc.PythonModel):
    """
    Production-grade inference pipeline.
    Input:  raw order features (pandas DataFrame)
    Output: late probability + prediction + risk tier

    Wrapping in mlflow.pyfunc makes this deployable
    anywhere — batch, REST API, cloud — without changing code.
    """

    def __init__(self, model, feature_cols):
        self.model        = model
        self.feature_cols = feature_cols

    def predict(self, context, model_input: pd.DataFrame):
        X      = model_input[self.feature_cols].fillna(0)
        probas = self.model.predict_proba(X)[:, 1]
        preds  = (probas >= 0.5).astype(int)
        tiers  = pd.cut(
            probas,
            bins=[0, 0.3, 0.6, 0.8, 1.0],
            labels=["Low", "Medium", "High", "Critical"]
        )
        return pd.DataFrame({
            "late_probability": probas.round(4),
            "predicted_late":   preds,
            "risk_tier":        tiers,
        })

pipeline = LateDeliveryPipeline(champion, FEATURES)
print("Pipeline class created!")

# Quick test — run 5 sample orders through it
df_silver = pd.read_csv("/content/drive/MyDrive/olist_project/silver/features.csv")
sample    = df_silver[FEATURES].head(5)
result    = pipeline.predict(None, sample)
print("\n Sample predictions:")
print(result)

Pipeline class created!

 Sample predictions:
   late_probability  predicted_late risk_tier
0            0.0006               0       Low
1            0.0216               0       Low
2            0.0229               0       Low
3            0.0472               0       Low
4            0.0496               0       Low


In [6]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Reload test set for final metrics
X = df_silver[FEATURES].fillna(0)
y = df_silver["target_late"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preds  = champion.predict(X_test)
probas = champion.predict_proba(X_test)[:, 1]

final_metrics = {
    "accuracy": round(accuracy_score(y_test, preds), 4),
    "f1":       round(f1_score(y_test, preds), 4),
    "roc_auc":  round(roc_auc_score(y_test, probas), 4),
}

# Log and register the full pipeline
with mlflow.start_run(run_name="LightGBM-SMOTE-PIPELINE-v1"):
    mlflow.log_params({
        "model_type":    "LightGBM",
        "balancing":     "SMOTE",
        "tuning":        "Optuna-50-trials",
        "n_features":    len(FEATURES),
        "train_rows":    len(X_train),
        "late_rate":     round(y.mean(), 4),
    })
    mlflow.log_metrics(final_metrics)

    # Register as versioned model
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=pipeline,
        registered_model_name="olist-late-delivery-predictor",
        pip_requirements=["lightgbm", "pandas", "numpy", "scikit-learn"]
    )

    run_id = mlflow.active_run().info.run_id
    print(" Pipeline registered in MLflow Model Registry!")
    print(f"   Model: olist-late-delivery-predictor")
    print(f"   Run ID: {run_id}")
    for k, v in final_metrics.items():
        print(f"   {k}: {v}")

2026/03/10 18:41:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 18:41:07 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/03/10 18:41:07 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.
/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///

 Pipeline registered in MLflow Model Registry!
   Model: olist-late-delivery-predictor
   Run ID: b8442c22596a45c48feda5412e3d7455
   accuracy: 0.9378
   f1: 0.5446
   roc_auc: 0.8985


In [7]:
# This is what runs nightly in production
# Load model FROM registry — not from file — that's the MLE pattern

model_uri     = "models:/olist-late-delivery-predictor/1"
loaded_model  = mlflow.pyfunc.load_model(model_uri)

# Score ALL orders
print("🔄 Running batch scoring on all 96,470 orders...")
predictions = loaded_model.predict(df_silver[FEATURES])

# Build Gold layer
df_gold = df_silver[["order_id", "target_late",
                      "review_comment_message",
                      "product_category_name_english"]].copy()
df_gold = pd.concat([
    df_gold.reset_index(drop=True),
    predictions.reset_index(drop=True)
], axis=1)

# Add business impact column
AVG_LATE_COST             = 25  # USD per late order (reverse logistics + support)
df_gold["revenue_at_risk"] = (df_gold["late_probability"] * AVG_LATE_COST).round(2)

# Summary
total_flagged  = df_gold[df_gold["predicted_late"] == 1]["order_id"].count()
critical       = (df_gold["risk_tier"] == "Critical").sum()
total_at_risk  = df_gold[df_gold["predicted_late"] == 1]["revenue_at_risk"].sum()

print(f"\n Batch scoring complete!")
print(f"   Total orders scored:     {len(df_gold):,}")
print(f"   Predicted late:          {total_flagged:,}")
print(f"   Critical risk orders:    {critical:,}")
print(f"   Total revenue at risk:   ${total_at_risk:,.0f}")
print(f"\n Risk tier breakdown:")
print(df_gold["risk_tier"].value_counts())

# Save Gold layer
os.makedirs("/content/drive/MyDrive/olist_project/gold", exist_ok=True)
df_gold.to_csv("/content/drive/MyDrive/olist_project/gold/predictions.csv", index=False)
print(f"\n Gold layer saved to Google Drive!")

🔄 Running batch scoring on all 96,470 orders...

 Batch scoring complete!
   Total orders scored:     96,470
   Predicted late:          5,293
   Critical risk orders:    1,745
   Total revenue at risk:   $96,132

 Risk tier breakdown:
risk_tier
Low         87595
Medium       4853
High         2277
Critical     1745
Name: count, dtype: int64

 Gold layer saved to Google Drive!
